# Bronze to Silver Transformation
Reads raw Parquet files from the bronze layer, cleans and validates the data, adds sector classification, and writes a Delta table to the silver layer.

In [0]:
# Read all bronze parquet files into a single DataFrame
bronze_df = spark.read.parquet("/Volumes/stock_pipeline/bronze/stock_data/*/*/*.parquet")

print(f"Total rows: {bronze_df.count()}")
print(f"Columns: {bronze_df.columns}")
bronze_df.show(5)

## Sector Mapping
Define sector classifications for all 37 tickers using a Spark map for distributed processing.

In [0]:
from pyspark.sql import functions as F

sector_map = {
    "META": "FAANG", "AAPL": "FAANG",
    "AMZN": "FAANG", "NFLX": "FAANG",
    "GOOGL": "FAANG", "MSFT": "Big Tech",
    "NVDA": "Big Tech", "AMD": "Big Tech",
    "INTC": "Big Tech", "TSLA": "Big Tech",
    "JPM": "Finance", "GS": "Finance", "BAC": "Finance",
    "MS": "Finance", "V": "Finance", "MA": "Finance",
    "AXP": "Finance", "BLK": "Finance",
    "JNJ": "Healthcare", "PFE": "Healthcare", "UNH": "Healthcare",
    "ABBV": "Healthcare", "MRK": "Healthcare",
    "WMT": "Retail", "TGT": "Retail", "NKE": "Retail",
    "MCD": "Retail", "SBUX": "Retail", "COST": "Retail",
    "XOM": "Energy", "CVX": "Energy", "COP": "Energy",
    "SPY": "ETF", "QQQ": "ETF", "DIA": "ETF",
    "IWM": "ETF", "VTI": "ETF"
}

mapping_expr = F.create_map([F.lit(x) for pair in sector_map.items() for x in pair])
print("Sector mapping created")

## Silver Transformation
Clean and enrich the bronze data — cast data types, round prices, add sector column.

In [0]:
silver_df = bronze_df \
    .withColumn("Date", F.col("Date").cast("date")) \
    .withColumn("Sector", mapping_expr[F.col("Ticker")]) \
    .withColumn("Open", F.round(F.col("Open"), 2)) \
    .withColumn("High", F.round(F.col("High"), 2)) \
    .withColumn("Low", F.round(F.col("Low"), 2)) \
    .withColumn("Close", F.round(F.col("Close"), 2)) \
    .withColumn("Volume", F.col("Volume").cast("integer"))

silver_df.show(5)

## Data Quality Checks
Remove duplicates and validate no nulls in critical columns before writing to silver.

In [0]:
# Remove duplicates
silver_df = silver_df.dropDuplicates(["Date", "Ticker"])

# Validate no nulls in critical columns
null_check = silver_df.filter(
    F.col("Close").isNull() | 
    F.col("Ticker").isNull() | 
    F.col("Date").isNull()
).count()

print(f"Rows with nulls in critical columns: {null_check}")
print(f"Total rows after dedup: {silver_df.count()}")

## Write to Silver Delta Table
Write the cleaned DataFrame as a managed Delta table in the silver schema.

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("stock_pipeline.silver.stock_prices")

print("Silver table written successfully")

In [0]:
%sql
SELECT * FROM stock_pipeline.silver.stock_prices
LIMIT 10

In [0]:
%sql
DESCRIBE TABLE stock_pipeline.silver.stock_prices

## SQL Equivalent - Bronze to Silver Transformation

```sql
CREATE OR REPLACE TABLE stock_pipeline.silver.stock_prices
USING DELTA AS
SELECT
    CAST(Date AS DATE)                    AS Date,
    Ticker,
    ROUND(Open, 2)                        AS Open,
    ROUND(High, 2)                        AS High,
    ROUND(Low, 2)                         AS Low,
    ROUND(Close, 2)                       AS Close,
    CAST(Volume AS INT)                   AS Volume,
    CASE Ticker
        WHEN 'META'  THEN 'FAANG'
        WHEN 'AAPL'  THEN 'FAANG'
        WHEN 'AMZN'  THEN 'FAANG'
        WHEN 'NFLX'  THEN 'FAANG'
        WHEN 'GOOGL' THEN 'FAANG'
        WHEN 'MSFT'  THEN 'Big Tech'
        WHEN 'NVDA'  THEN 'Big Tech'
        WHEN 'AMD'   THEN 'Big Tech'
        WHEN 'INTC'  THEN 'Big Tech'
        WHEN 'TSLA'  THEN 'Big Tech'
        WHEN 'JPM'   THEN 'Finance'
        WHEN 'GS'    THEN 'Finance'
        WHEN 'BAC'   THEN 'Finance'
        WHEN 'MS'    THEN 'Finance'
        WHEN 'V'     THEN 'Finance'
        WHEN 'MA'    THEN 'Finance'
        WHEN 'AXP'   THEN 'Finance'
        WHEN 'BLK'   THEN 'Finance'
        WHEN 'JNJ'   THEN 'Healthcare'
        WHEN 'PFE'   THEN 'Healthcare'
        WHEN 'UNH'   THEN 'Healthcare'
        WHEN 'ABBV'  THEN 'Healthcare'
        WHEN 'MRK'   THEN 'Healthcare'
        WHEN 'WMT'   THEN 'Retail'
        WHEN 'TGT'   THEN 'Retail'
        WHEN 'NKE'   THEN 'Retail'
        WHEN 'MCD'   THEN 'Retail'
        WHEN 'SBUX'  THEN 'Retail'
        WHEN 'COST'  THEN 'Retail'
        WHEN 'XOM'   THEN 'Energy'
        WHEN 'CVX'   THEN 'Energy'
        WHEN 'COP'   THEN 'Energy'
        WHEN 'SPY'   THEN 'ETF'
        WHEN 'QQQ'   THEN 'ETF'
        WHEN 'DIA'   THEN 'ETF'
        WHEN 'IWM'   THEN 'ETF'
        WHEN 'VTI'   THEN 'ETF'
        ELSE 'Unknown'
    END AS Sector
FROM parquet.`/Volumes/stock_pipeline/bronze/stock_data/*/*/*.parquet`
WHERE Close IS NOT NULL
  AND Ticker IS NOT NULL
  AND Date IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY Date, Ticker ORDER BY Date) = 1
```